# 4. 모델 성능 평가

**성능 수치는 확정값이 아니다.** 검증 구간의 실제 불량 로트가 5건 안팎이라 한 건 차이로 크게 움직인다.

평가 절차는 다음과 같이 고정한다. **어느 항목도 결과를 보고 바꾸지 않는다.**

| 항목 | 값 |
|---|---|
| 검증 | 시간순 — 학습 구간을 한 로트씩 늘리며 나머지로 검증 |
| 집계 대상 | 학습 구간 양성 **25건 이상**인 분할만 |
| 임계값 | 학습 구간 양성비율에서 산출 (검증 미참조) |
| 주 지표 | 재현율 · F1 · 정확도 (기준선 대비 병기) |

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/data/Project/boyoung/pipeline')
for v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(v, '1')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
sns.set_theme(style='whitegrid', font='NanumGothic', rc={'axes.unicode_minus': False})
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)

import db, preprocess as P, classify as C

data, report = P.build()
data = data.sort_values('target_date').reset_index(drop=True)
th = C.threshold(data['bad_rate']); y = C.label(data['bad_rate'], th); yv = y.values
print(f'{len(data)}로트  양성 {int(yv.sum())}  판정 기준 {th:.2f}%')

## 4.1 왜 시간순인가

목적이 '지금 학습해 앞으로 들어올 로트를 판정하는 것'이므로, 그 조건을 그대로 흉내내야 한다.

In [ ]:
# 공정 정보가 전혀 없는 '달 번호' 하나만으로 무작위 교차검증을 돌려 본다
for name, cols in [('달 번호 하나만', ['month']), ('모래온도', ['sand_temp_mean']),
                   ('벤토나이트', ['bentonite'])]:
    s, _ = C.evaluate(data[cols], yv)
    b = s[s['모델'] != C.BASELINE]
    print(f'  {name:12} 정확도 {b["정확도"].max():.3f}   F1 {b["F1"].max():.3f}')

**달 번호만으로 0.71 이 나온다.** 무작위 분할 성적의 상당 부분이 계절 암기다.

## 4.2 시간순 검증

In [ ]:
r = C.timed(data[['bentonite']], y, min_train=25, min_pos=1)
r['학습양성'] = [int(yv[:n].sum()) for n in r['학습']]
sel = r[r['학습양성'] >= 25]

print(f'전체 분할 {len(r)}   집계 대상 {len(sel)} (학습 {int(sel["학습"].min())}~{int(sel["학습"].max())}로트)\n')
print(sel[['정확도','F1','재현율','정밀도','임계값']].agg(['mean','std','min','max']).round(3).to_string())

### 집계 조건을 바꾸면

**학습 구간 양성 25건 이상**은 학습 데이터만 보고 정할 수 있는 조건이라 누수가 아니다.

In [ ]:
rows = []
for k, n in [(0,'조건 없음'), (20,'학습 양성 20건 이상'), (25,'학습 양성 25건 이상')]:
    s = r[r['학습양성'] >= k]
    rows.append({'조건': n, '분할': len(s), '정확도': s['정확도'].mean(), 'F1': s['F1'].mean(),
                 '재현율': s['재현율'].mean(), '정밀도': s['정밀도'].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\n현재 93로트에 양성 32건이므로 이미 마지막 조건 안에 들어와 있고, 앞으로 줄어들지 않는다.')

## 4.3 평균을 그대로 읽으면 안 된다

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for c, col in [('F1','#b8531c'), ('재현율','#0b6b6b'), ('정확도','#7aa6c2')]:
    ax[0].plot(sel['학습'], sel[c], marker='o', ms=3.5, lw=1.6, color=col, label=c)
ax[0].set_xlabel('학습에 쓴 로트 수'); ax[0].set_ylim(0, 1)
ax[0].legend(fontsize=8, ncol=3); ax[0].set_title('분할별 성능')
sns.histplot(sel['F1'], bins=12, ax=ax[1], color='#b8531c')
ax[1].set_xlabel('F1'); ax[1].set_ylabel('분할 수'); ax[1].set_title('F1 은 두 덩어리로 갈린다')
plt.tight_layout(); plt.show()

print(sel[['학습','검증','양성','임계값','정확도','F1','재현율','정밀도']].round(3).to_string(index=False))

**F1 은 0.667 아니면 0.000 이다.** 평균 0.531 은 실제로 나온 적 없는 값이다.

0 인 분할은 **검증 구간 양성이 1~5건**인 경우로, 한 건만 놓쳐도 0 이 된다.
또 학습 63~78 의 18개 분할이 같은 값인 것은 **같은 불량 로트 5건을 반복해서 재기 때문**이다 — 분할들은 서로 독립이 아니다.

## 4.4 기준선

기준선은 아무것도 학습하지 않고 다수 쪽만 찍는 모델의 정확도다. 양성 비율에 따라 달라지므로 **실측과 같은 구간에서 계산한다.**

In [ ]:
base_w = [max(yv[n:].mean(), 1-yv[n:].mean()) for n in sel['학습']]
print(f'검증 구간 기준선 (분할 평균)  {np.mean(base_w):.3f}')
print(f'실측 정확도               {sel["정확도"].mean():.3f}')

검증 구간은 뒤쪽(7~9월)이라 불량이 적어 **기준선 자체가 높다.** 전체 기준선과 비교하면 개선 폭이 부풀려진다.

## 4.5 혼동행렬

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

cut = C.cut_rule(data[['bentonite']][:70], y[:70])
m = make_pipeline(C.impute(['bentonite']),
                  C.candidates(0,'balanced')['xgboost']).fit(data[['bentonite']][:70], y[:70])
prob = m.predict_proba(data[['bentonite']][70:])[:, 1]
pred = (prob >= cut).astype(int)
cm = confusion_matrix(yv[70:], pred)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
labs = ['정확도','F1','재현율','정밀도']; vals = [sel[c].mean() for c in labs]
b = ax[0].bar(labs, vals, color=['#7aa6c2','#b8531c','#0b6b6b','#9a7aa8'], width=.6)
ax[0].axhline(np.mean(base_w), color='#333', ls='--', lw=1.4, label=f'기준선 {np.mean(base_w):.3f}')
ax[0].set_ylim(0,1); ax[0].legend(fontsize=9); ax[0].set_title('시간순 성능')
for r_, v in zip(b, vals):
    ax[0].text(r_.get_x()+r_.get_width()/2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, cbar=False, ax=ax[1],
            xticklabels=['정상 예측','불량 예측'], yticklabels=['정상 실제','불량 실제'],
            annot_kws={'size':15})
ax[1].set_title(f'학습 70 · 검증 {len(data)-70}로트')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'놓친 불량 {fn} / {tp+fn}    헛경보 {fp} / {tn+fp}')

## 4.6 도달 가능한 상한

벤토나이트 8종을 각각 불량/정상으로 라벨링하는 256가지 중 **검증 구간에서 최선인 조합**을 고른다. 답을 미리 보고 고르는 오라클이므로 실제보다 후하다.

In [ ]:
import itertools
from sklearn.metrics import accuracy_score, f1_score

b_ = data['bentonite'].fillna(data['bentonite'].median()).values
lv = sorted(np.unique(b_))
POS = [n for n in range(25, len(yv)) if yv[:n].sum() >= 25]

acc_c, f1_c = [], []
for n in POS:
    te = np.arange(n, len(yv))
    if yv[te].sum() < 1: continue
    ba = bf = -1
    for pick in itertools.product([0,1], repeat=len(lv)):
        mp = dict(zip(lv, pick)); p_ = np.array([mp[x] for x in b_[te]])
        ba = max(ba, accuracy_score(yv[te], p_)); bf = max(bf, f1_score(yv[te], p_, zero_division=0))
    acc_c.append(ba); f1_c.append(bf)

print(f'{"":18}{"정확도":>9}{"F1":>9}')
print(f'{"기준선":18}{np.mean(base_w):9.3f}{0.0:9.3f}')
print(f'{"실측":18}{sel["정확도"].mean():9.3f}{sel["F1"].mean():9.3f}')
print(f'{"상한 (오라클)":18}{np.mean(acc_c):9.3f}{np.mean(f1_c):9.3f}')
print(f'\n남은 여유  정확도 {np.mean(acc_c)-sel["정확도"].mean():+.3f}   F1 {np.mean(f1_c)-sel["F1"].mean():+.3f}')

**모델을 더 손봐서 얻을 것이 거의 없다.** 같은 벤토나이트 값 안에서 불량과 정상이 섞여 있어 가릴 방법이 없기 때문이다.

## 4.7 예측 결과 확인

In [ ]:
import predict
t = predict.by_value()
print(f'판정 기준 불량률 > {t.attrs["rate_cut"]:.2f}%   위험확률 임계값 {t.attrs["prob_cut"]:.3f}\n')
print(t.to_string(index=False))

In [ ]:
rv = predict.review()
print('학습 로트 되짚기 — 성능 평가가 아니다\n')
print(rv['결과'].value_counts().to_frame('로트').to_string())
print()
print(rv[rv['결과'] != '맞음'].drop(columns='결과').to_string(index=False))

**놓친 로트 대부분이 벤토나이트 28·29 다.** 값이 높은데 불량률이 10% 를 넘는 로트들로, 벤토나이트만으로는 설명할 수 없다.

헛경보는 대부분 25·26 이고 실제 불량률이 판정 기준 바로 아래에 몰려 있다.

**같은 벤토나이트 값 안에서 무엇이 달랐는지 설명할 변수가 필요하다** — 주입 온도 · 냉각 조건 · 금형 · 용해로 성분.

## 4.8 정리

| | |
|---|---|
| 정확도 | 0.796 (검증 구간 기준선 0.754 대비 +4.2%p) |
| F1 | 0.531 — 0.667 과 0.000 의 평균이며 대표값이 아니다 |
| 재현율 | 0.622 |
| 정밀도 | 0.465 |

**확정할 수 없는 이유** — 검증 구간의 실제 불량 로트가 5건 안팎이다. 한 건 차이로 재현율이 0.60~1.00 을 오간다.

**다음 측정** — 매월 10일 전후 전월 불량일보가 반영되면 위 절차 그대로 한 번 측정하고 기록한다. 수치가 나쁘게 나와도 절차를 고치지 않는다.